# Flickr30k T3→S3 ONNX FP32 Export & Parity

Xuất checkpoint triển khai seed 42 đã khóa sang hai graph ONNX FP32 và kiểm
tra parity với PyTorch bằng dữ liệu tổng hợp. Notebook không đọc Train,
Validation, ảnh Flickr30k hoặc Held-out Test.

## Kaggle input bắt buộc — đúng 4 input

1. Output `F30K_T3S3_Final_Lock`.
2. Output `F30K_T3S3_LogitKD_42`.
3. Output `F30K_Data`.
4. Dataset `ONNX_Runtime_Python313_Offline`.

Runtime: CPU/Accelerator None, Internet Off, dự kiến 5–20 phút. Decoder sử
dụng buffer cố định 31 token để tránh graph động và phù hợp vòng lặp sinh
caption trên Android.

## Cell 1 — Integrity, selected checkpoint and offline runtime

In [1]:
import hashlib
import importlib.util
import json
import os
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

EXPERIMENT_ID = "F30K_T3S3_ONNX_FP32"
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(x) for x in matches])
    return matches[0]

LOCK_PATH = unique_named("F30K_T3S3_FINAL_METHOD_LOCKED.json")
KD_RESULT_PATH = unique_named("F30K_T3S3_LogitKD_42_RESULT.json")
VOCAB_PATH = unique_named("f30k_vocabulary_minfreq5.json")
lock = read_json(LOCK_PATH)
kd_result = read_json(KD_RESULT_PATH)
vocab = read_json(VOCAB_PATH)

assert lock["status"] == "locked"
assert lock["method"] == "Direct Logit-KD"
assert lock["deployment_checkpoint_selection"]["seed"] == 42
assert lock["held_out_test_authorized_now"] is False
assert lock["test_set_status"] == "sealed"
assert lock["test_set_used"] is False
assert kd_result["status"] == "complete"
assert kd_result["experiment_id"] == "F30K_T3S3_LogitKD_42"
assert kd_result["student_id"] == "S3"
assert kd_result["seed"] == 42
assert kd_result["test_set_status"] == "sealed"
assert kd_result["test_set_used"] is False
assert len(vocab["tokens"]) == 7456

BEST_PATH = KD_RESULT_PATH.parent / "best.pt"
assert sha256(BEST_PATH) == kd_result["artifacts"]["best.pt"]["sha256"]
assert sha256(BEST_PATH) == lock["deployment_checkpoint_selection"][
    "best_checkpoint_sha256"
]

if importlib.util.find_spec("onnxruntime") is None:
    wheels = sorted(INPUT_ROOT.rglob("onnxruntime*.whl"))
    assert wheels, "Hãy gắn dataset ONNX_Runtime_Python313_Offline."
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", str(wheels[0])]
    )
assert importlib.util.find_spec("onnx") is not None, "Kaggle runtime thiếu onnx."
assert importlib.util.find_spec("onnxruntime") is not None

import onnx
import onnxruntime as ort
import timm

torch.manual_seed(20261026)
np.random.seed(20261026)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
print("Preflight              : PASSED")
print("Selected checkpoint    : seed 42")
print("Checkpoint SHA-256     :", sha256(BEST_PATH))
print("Vocabulary             :", len(vocab["tokens"]))
print("ONNX / ONNX Runtime    :", onnx.__version__, "/", ort.__version__)
print("Validation / Test read : False / False")

Preflight              : PASSED
Selected checkpoint    : seed 42
Checkpoint SHA-256     : 862f16b5b08bf580eac13ac112d529e9beeac7b8420116c598210f5b2c3ea4b4
Vocabulary             : 7456
ONNX / ONNX Runtime    : 1.22.0 / 1.28.0
Validation / Test read : False / False


## Cell 2 — Rebuild S3 and strict-load the locked checkpoint

In [2]:
MODEL_ID = "S3"
VOCABULARY_SIZE = 7456
MAX_STEPS = 31

def select_tensor(value):
    if torch.is_tensor(value):
        return value
    if isinstance(value, dict):
        return [item for item in value.values() if torch.is_tensor(item)][-1]
    if isinstance(value, (tuple, list)):
        return [item for item in value if torch.is_tensor(item)][-1]
    raise TypeError(type(value))

class VisualEncoder(nn.Module):
    def __init__(self, backbone, source_dim):
        super().__init__()
        self.backbone = backbone
        self.source_dim = source_dim
        self.projection = nn.Linear(source_dim, 256)
        self.visual_position_embedding = nn.Parameter(torch.empty(1, 49, 256))
        self.output_norm = nn.LayerNorm(256)
    def forward(self, images):
        features = select_tensor(self.backbone.forward_features(images))
        tokens = features.flatten(2).transpose(1, 2).contiguous()
        return self.output_norm(
            self.projection(tokens) + self.visual_position_embedding
        )

class CaptionDecoder(nn.Module):
    def __init__(self, vocabulary_size):
        super().__init__()
        self.vocabulary_size = vocabulary_size
        self.d_model, self.max_steps = 256, MAX_STEPS
        self.token_embedding = nn.Embedding(vocabulary_size, 256, padding_idx=0)
        self.position_embedding = nn.Embedding(MAX_STEPS, 256)
        self.embedding_dropout = nn.Dropout(0.1)
        layer = nn.TransformerDecoderLayer(
            256, 8, 1024, 0.1, "gelu",
            batch_first=True, norm_first=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(layer, 3)
        self.final_norm = nn.LayerNorm(256)
        self.output_bias = nn.Parameter(torch.zeros(vocabulary_size))
    def forward(self, token_ids, memory, attention_mask):
        positions = torch.arange(
            MAX_STEPS, device=token_ids.device
        ).unsqueeze(0)
        hidden = self.embedding_dropout(
            self.token_embedding(token_ids) * 16.0
            + self.position_embedding(positions)
        )
        causal = torch.triu(
            torch.ones(
                MAX_STEPS, MAX_STEPS,
                dtype=torch.bool, device=token_ids.device,
            ),
            diagonal=1,
        )
        hidden = self.transformer_decoder(
            hidden,
            memory,
            tgt_mask=causal,
            tgt_key_padding_mask=~attention_mask.bool(),
        )
        return F.linear(
            self.final_norm(hidden),
            self.token_embedding.weight,
            self.output_bias,
        )

backbone = timm.create_model(
    "tiny_vit_5m_224.dist_in22k_ft_in1k", pretrained=False
)
backbone.reset_classifier(num_classes=0, global_pool="")
for head_name in ["head", "head_dist", "classifier", "fc"]:
    head = getattr(backbone, head_name, None)
    if isinstance(head, nn.Module) and not isinstance(head, nn.Identity):
        setattr(backbone, head_name, nn.Identity())

encoder = VisualEncoder(backbone, 320)
decoder = CaptionDecoder(VOCABULARY_SIZE)
checkpoint = torch.load(BEST_PATH, map_location="cpu", weights_only=False)
assert checkpoint["experiment_id"] == "F30K_T3S3_LogitKD_42"
encoder.load_state_dict(checkpoint["encoder_state_dict"], strict=True)
decoder.load_state_dict(checkpoint["decoder_state_dict"], strict=True)
encoder.eval()
decoder.eval()

encoder_parameters = sum(parameter.numel() for parameter in encoder.parameters())
decoder_parameters = sum(parameter.numel() for parameter in decoder.parameters())
assert encoder_parameters == 5_166_356
assert decoder_parameters == 5_084_960
print("Strict checkpoint load : PASSED")
print("Encoder parameters     :", encoder_parameters)
print("Decoder parameters     :", decoder_parameters)
print("Total parameters       :", encoder_parameters + decoder_parameters)
print("FP32 parameter estimate:",
      (encoder_parameters + decoder_parameters) * 4 / 1024**2, "MiB")

Strict checkpoint load : PASSED
Encoder parameters     : 5166356
Decoder parameters     : 5084960
Total parameters       : 10251316
FP32 parameter estimate: 39.10566711425781 MiB


## Cell 3 — Export encoder and fixed-shape decoder

In [3]:
ENCODER_ONNX = WORK_ROOT / "f30k_t3s3_seed42_encoder_fp32.onnx"
DECODER_ONNX = WORK_ROOT / "f30k_t3s3_seed42_decoder_fp32.onnx"

sample_image = torch.randn(1, 3, 224, 224, dtype=torch.float32)
sample_tokens = torch.zeros(1, MAX_STEPS, dtype=torch.long)
sample_tokens[0, :8] = torch.tensor([1, 25, 99, 301, 48, 76, 5, 2])
sample_mask = torch.zeros(1, MAX_STEPS, dtype=torch.bool)
sample_mask[0, :8] = True
# Không dùng inference_mode ở đây: tensor inference không thể được exporter
# decoder lưu cho quá trình tracing. no_grad vẫn tránh dựng gradient graph
# nhưng tạo tensor thường, tương thích torch.onnx.export.
with torch.no_grad():
    sample_memory = encoder(sample_image)
sample_memory = sample_memory.detach().clone()
assert tuple(sample_memory.shape) == (1, 49, 256)

started = time.perf_counter()
print("[ONNX] Exporting encoder...", flush=True)
torch.onnx.export(
    encoder,
    (sample_image,),
    str(ENCODER_ONNX),
    input_names=["images"],
    output_names=["memory"],
    opset_version=17,
    do_constant_folding=True,
    dynamo=False,
)
print("[ONNX] Encoder exported in",
      round(time.perf_counter() - started, 2), "s", flush=True)
onnx.checker.check_model(onnx.load(str(ENCODER_ONNX)))
print("[ONNX] Encoder checker: PASSED", flush=True)

started = time.perf_counter()
print("[ONNX] Exporting fixed-shape decoder...", flush=True)
torch.onnx.export(
    decoder,
    (sample_tokens, sample_memory, sample_mask),
    str(DECODER_ONNX),
    input_names=["token_ids", "memory", "attention_mask"],
    output_names=["logits"],
    opset_version=17,
    do_constant_folding=True,
    dynamo=False,
)
print("[ONNX] Decoder exported in",
      round(time.perf_counter() - started, 2), "s", flush=True)
onnx.checker.check_model(onnx.load(str(DECODER_ONNX)))
print("[ONNX] Decoder checker: PASSED", flush=True)

assert ENCODER_ONNX.stat().st_size > 1_000_000
assert DECODER_ONNX.stat().st_size > 1_000_000
print("Encoder / decoder MiB  :",
      ENCODER_ONNX.stat().st_size / 1024**2,
      "/", DECODER_ONNX.stat().st_size / 1024**2)

[ONNX] Exporting encoder...


/tmp/ipykernel_16/2412020736.py:19: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.13/dist-packages/timm/models/tiny_vit.py:390: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if H == self.window_size and W == self.window_size:
/usr/local/lib/python3.13/dist-packages/timm/models/tiny_vit.py:397: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't rec

[ONNX] Encoder exported in 1.82 s
[ONNX] Encoder checker: PASSED
[ONNX] Exporting fixed-shape decoder...


/tmp/ipykernel_16/2412020736.py:36: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


[ONNX] Decoder exported in 0.99 s
[ONNX] Decoder checker: PASSED
Encoder / decoder MiB  : 20.17105770111084 / 26.801671028137207


## Cell 4 — Five-case PyTorch↔ONNX Runtime parity and contract

In [4]:
encoder_session = ort.InferenceSession(
    str(ENCODER_ONNX), providers=["CPUExecutionProvider"]
)
decoder_session = ort.InferenceSession(
    str(DECODER_ONNX), providers=["CPUExecutionProvider"]
)

parity_rows = []
for case_index, valid_length in enumerate([1, 4, 8, 16, 30]):
    generator = torch.Generator().manual_seed(20261026 + case_index)
    image = torch.randn(1, 3, 224, 224, generator=generator)
    tokens = torch.zeros(1, MAX_STEPS, dtype=torch.long)
    tokens[0, 0] = 1
    if valid_length > 1:
        tokens[0, 1:valid_length] = torch.randint(
            4, VOCABULARY_SIZE,
            (valid_length - 1,), generator=generator,
        )
    mask = torch.zeros(1, MAX_STEPS, dtype=torch.bool)
    mask[0, :valid_length] = True
    with torch.inference_mode():
        torch_memory = encoder(image)
        torch_logits = decoder(tokens, torch_memory, mask)
    ort_memory = encoder_session.run(
        ["memory"], {"images": image.numpy()}
    )[0]
    ort_logits = decoder_session.run(
        ["logits"],
        {
            "token_ids": tokens.numpy(),
            "memory": ort_memory,
            "attention_mask": mask.numpy(),
        },
    )[0]
    torch_memory_np = torch_memory.numpy()
    torch_logits_np = torch_logits.numpy()
    memory_max_abs = float(np.max(np.abs(torch_memory_np - ort_memory)))
    valid_slice = slice(0, valid_length)
    logits_max_abs = float(np.max(np.abs(
        torch_logits_np[:, valid_slice] - ort_logits[:, valid_slice]
    )))
    torch_top1 = torch_logits_np[:, valid_length - 1].argmax(axis=-1)
    ort_top1 = ort_logits[:, valid_length - 1].argmax(axis=-1)
    top1_equal = bool(np.array_equal(torch_top1, ort_top1))
    assert np.isfinite(ort_memory).all()
    assert np.isfinite(ort_logits[:, valid_slice]).all()
    assert memory_max_abs <= 1e-4, memory_max_abs
    assert logits_max_abs <= 2e-4, logits_max_abs
    assert top1_equal
    parity_rows.append({
        "case": case_index,
        "valid_token_length": valid_length,
        "encoder_max_abs_error": memory_max_abs,
        "decoder_valid_logits_max_abs_error": logits_max_abs,
        "next_token_top1_equal": top1_equal,
    })

contract = {
    "schema_version": "f30k_t3s3_onnx_fp32_contract_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "passed",
    "experiment_id": EXPERIMENT_ID,
    "source": {
        "pair": "T3->S3",
        "method": "Direct Logit-KD",
        "seed": 42,
        "checkpoint_sha256": sha256(BEST_PATH),
        "final_lock_sha256": sha256(LOCK_PATH),
    },
    "format": "ONNX FP32",
    "opset": 17,
    "encoder_input": {"images": [1, 3, 224, 224], "dtype": "float32"},
    "encoder_output": {"memory": [1, 49, 256], "dtype": "float32"},
    "decoder_inputs": {
        "token_ids": [1, 31],
        "memory": [1, 49, 256],
        "attention_mask": [1, 31],
    },
    "decoder_output": {"logits": [1, 31, 7456], "dtype": "float32"},
    "mobile_decoding_policy": (
        "Fixed 31-token buffer; fill generated prefix, pad remainder with 0, "
        "and mark the valid prefix in attention_mask."
    ),
    "parity": parity_rows,
    "artifacts": {
        ENCODER_ONNX.name: {
            "bytes": ENCODER_ONNX.stat().st_size,
            "sha256": sha256(ENCODER_ONNX),
        },
        DECODER_ONNX.name: {
            "bytes": DECODER_ONNX.stat().st_size,
            "sha256": sha256(DECODER_ONNX),
        },
    },
    "validation_data_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "next_phase_authorized": "F30K_T3S3_INT8_PREFLIGHT",
}
contract_path = WORK_ROOT / "F30K_T3S3_ONNX_FP32_CONTRACT.json"
result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
write_json(contract_path, contract)
write_json(result_path, contract)

manifest_artifacts = []
for path in [ENCODER_ONNX, DECODER_ONNX, contract_path, result_path]:
    manifest_artifacts.append({
        "relative_path": path.name,
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    })
write_json(WORK_ROOT / "MANIFEST.json", {
    "schema_version": "f30k_t3s3_onnx_fp32_manifest_v1",
    "experiment_id": EXPERIMENT_ID,
    "status": "complete",
    "artifacts": manifest_artifacts,
    "validation_data_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})
print("Parity cases           :", len(parity_rows), "/", len(parity_rows),
      "PASSED")
print("Worst encoder error    :",
      max(row["encoder_max_abs_error"] for row in parity_rows))
print("Worst decoder error    :",
      max(row["decoder_valid_logits_max_abs_error"] for row in parity_rows))
print("Status                 : COMPLETE")
print("Held-out Test          : SEALED / NOT USED")
print("Next                   :", contract["next_phase_authorized"])

Parity cases           : 5 / 5 PASSED
Worst encoder error    : 1.990795135498047e-05
Worst decoder error    : 1.0967254638671875e-05
Status                 : COMPLETE
Held-out Test          : SEALED / NOT USED
Next                   : F30K_T3S3_INT8_PREFLIGHT
